# INT221 - Basketball Team Performance Analysis
### Manu Bansal

**Objective:** Big East college basketball teams and analysts use historical team and game statistics to evaluate performance and understand what drives winning seasons. Identify the key performance drivers to estimate win percentage, rankings, or season success and support data-driven coaching and recruitment decisions.

**This notebook covers:**
1. Data Loading & Understanding
2. Data Wrangling (Null handling, Duplicates, Outliers)
3. Feature Engineering
4. Encoding Categorical Variables
5. Exploratory Data Analysis (EDA)
6. Hypothesis Framing & Testing
7. Classification Model & Confusion Matrix
8. Final Observations

---
## 1. Installing & Importing Libraries

In [ ]:
pip install matplotlib pandas numpy seaborn scikit-learn scipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, ttest_ind, ttest_rel, chi2_contingency, spearmanr
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay, accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

---
## 2. Loading the Dataset

In [ ]:
df = pd.read_csv("11_Baseketball_Team_Performance_Analysis.csv")
print(f"Dataset Shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

### 2.1 Understanding the Dataset Structure
Let's check data types, non-null counts, and basic statistics.

In [ ]:
df.info()

In [ ]:
# Column names and data types overview
print("Column Names:")
print(df.columns.tolist())
print(f"\nUnique Schools: {df['school'].nunique()}")
print(f"Year Range: {df['year'].min()} - {df['year'].max()}")

---
## 3. Data Wrangling

### 3.1 Checking & Handling NULL Values

In [ ]:
# Count null values per column
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
null_df = pd.DataFrame({'Null Count': null_counts, 'Null %': null_pct})
print("Columns with NULL values:")
null_df[null_df['Null Count'] > 0]

**Observation:** Several columns have missing values (home_wins, away_wins, offensive_rating, 3_pointers, etc.).

**Strategy:** Fill numeric nulls with **column mean** since these are continuous stats with roughly symmetric distributions. Mean imputation preserves the overall distribution shape.

In [ ]:
# Fill null values with column mean for numeric columns
df = df.fillna(df.mean(numeric_only=True))

# Verify no nulls remain
print("Null values after handling:")
print(df.isnull().sum().sum(), "total nulls remaining")

### 3.2 Checking & Removing Duplicates

In [ ]:
print(f"Duplicate rows found: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Shape after removing duplicates: {df.shape}")

### 3.3 Dropping the `id` Column
The `id` column is just a row identifier with no analytical value. Keeping it would pollute correlations.

In [ ]:
df = df.drop(columns=['id'])
print(f"Shape after dropping 'id': {df.shape}")
df.head()

### 3.4 Descriptive Statistics

In [ ]:
df.describe()

---
## 4. Outlier Detection & Handling

### 4.1 Visualizing Outliers using Boxplots

In [ ]:
# Boxplots for key performance columns
cols_to_check = ['win_percentage', 'points', 'opponent_points', 'field_goal_percentage',
                 'simple_rating', 'assists', 'turnovers', 'total_rebounds']

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle('Boxplots - Outlier Detection (Before Capping)', fontsize=14, fontweight='bold')

for i, col in enumerate(cols_to_check):
    row, col_idx = i // 4, i % 4
    sns.boxplot(y=df[col], ax=axes[row][col_idx], color='skyblue')
    axes[row][col_idx].set_title(col, fontsize=10)

plt.tight_layout()
plt.show()

### 4.2 Counting Outliers using IQR Method

In [ ]:
# Count outliers per column using IQR
def count_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return ((df[column] < lower) | (df[column] > upper)).sum()

numeric_cols = df.select_dtypes(include='number').columns
outlier_counts = {col: count_outliers(df, col) for col in numeric_cols}
outlier_df = pd.DataFrame.from_dict(outlier_counts, orient='index', columns=['Outlier Count'])
outlier_df = outlier_df[outlier_df['Outlier Count'] > 0].sort_values('Outlier Count', ascending=False)
print("Columns with outliers (IQR method):")
outlier_df

### 4.3 Capping Outliers (Winsorization)
**Why capping instead of removing?** In sports data, extreme values are often genuine (e.g., a dominant team). Capping reduces their impact without losing data rows.

In [ ]:
# Cap outliers using IQR bounds
def cap_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    df[column] = df[column].clip(lower=Q1 - 1.5 * IQR, upper=Q3 + 1.5 * IQR)
    return df

for col in numeric_cols:
    df = cap_outliers(df, col)

print("Outliers capped successfully!")
print(f"Dataset shape remains: {df.shape}")

In [ ]:
# Boxplots AFTER capping to verify
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle('Boxplots - After Outlier Capping', fontsize=14, fontweight='bold')

cols_to_check = ['win_percentage', 'points', 'opponent_points', 'field_goal_percentage',
                 'simple_rating', 'assists', 'turnovers', 'total_rebounds']

for i, col in enumerate(cols_to_check):
    row, col_idx = i // 4, i % 4
    sns.boxplot(y=df[col], ax=axes[row][col_idx], color='lightgreen')
    axes[row][col_idx].set_title(col, fontsize=10)

plt.tight_layout()
plt.show()

---
## 5. Feature Engineering
Creating new meaningful features that capture performance insights beyond raw stats.

In [ ]:
# Point Differential - core measure of team dominance
df['point_diff'] = df['points'] - df['opponent_points']

# Home Win Percentage - measures home court advantage
df['home_win_pct'] = df['home_wins'] / (df['home_wins'] + df['home_losses'])

# Away Win Percentage - measures road performance
df['away_win_pct'] = df['away_wins'] / (df['away_wins'] + df['away_losses'])

# Conference Win Percentage - conference strength indicator
df['conference_win_pct'] = df['conference_wins'] / (df['conference_wins'] + df['conference_losses'])

# Assist-to-Turnover Ratio - ball handling efficiency
df['assist_turnover_ratio'] = df['assists'] / df['turnovers']

# Win Category - binary label for classification (above/below median win%)
median_win_pct = df['win_percentage'].median()
df['win_category'] = df['win_percentage'].apply(lambda x: 'High' if x >= median_win_pct else 'Low')

print("New features created:")
print(['point_diff', 'home_win_pct', 'away_win_pct', 'conference_win_pct', 'assist_turnover_ratio', 'win_category'])
df[['school', 'win_percentage', 'point_diff', 'home_win_pct', 'away_win_pct', 
    'conference_win_pct', 'assist_turnover_ratio', 'win_category']].head(10)

---
## 6. Encoding Categorical Variables

### 6.1 Label Encoding for `school`
Label Encoding assigns a unique number to each school. We keep the original column for reference.

In [ ]:
le = LabelEncoder()
df['school_encoded'] = le.fit_transform(df['school'])

print("Label Encoding Mapping:")
encoding_map = df[['school', 'school_encoded']].drop_duplicates().sort_values('school_encoded')
encoding_map

### 6.2 One-Hot Encoding for `win_category`
One-Hot encoding creates binary columns — useful for classification.

In [ ]:
# One-Hot Encode win_category
df_encoded = pd.get_dummies(df, columns=['win_category'], prefix='win_cat', drop_first=False)
print("One-Hot Encoded columns added:")
print([c for c in df_encoded.columns if 'win_cat' in c])
df_encoded[[c for c in df_encoded.columns if 'win_cat' in c]].head()

---
## 7. Exploratory Data Analysis (EDA)

### 7.1 Distribution of Win Percentage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
sns.histplot(df['win_percentage'], bins=30, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Distribution of Win Percentage', fontweight='bold')
axes[0].axvline(df['win_percentage'].mean(), color='red', linestyle='--', label=f"Mean: {df['win_percentage'].mean():.3f}")
axes[0].legend()

# Points distribution
sns.histplot(df['points'], bins=30, kde=True, color='coral', ax=axes[1])
axes[1].set_title('Distribution of Points Per Game', fontweight='bold')
axes[1].axvline(df['points'].mean(), color='red', linestyle='--', label=f"Mean: {df['points'].mean():.1f}")
axes[1].legend()

plt.tight_layout()
plt.show()

### 7.2 Correlation Heatmap
Identifying which stats are most strongly correlated with win percentage.

In [ ]:
# Select key columns for heatmap
key_cols = ['win_percentage', 'points', 'opponent_points', 'field_goal_percentage',
            'simple_rating', 'assists', 'turnovers', 'total_rebounds', 'steals',
            'blocks', 'offensive_rating', 'defensive_rating', 'net_rating',
            'point_diff', 'assist_turnover_ratio']

corr_matrix = df[key_cols].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Heatmap - Key Performance Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Key Observations from Heatmap:**
- `simple_rating`, `net_rating`, and `point_diff` are strongly correlated with `win_percentage`
- `field_goal_percentage` has a positive correlation with winning
- `turnovers` tends to have a negative impact on performance

### 7.3 Pairplot - Key Variables

In [ ]:
# Pairplot for key performance indicators
pair_cols = ['win_percentage', 'points', 'field_goal_percentage', 'simple_rating', 'assists']
sns.pairplot(df[pair_cols + ['win_category']], hue='win_category', 
             palette={'High': 'green', 'Low': 'red'}, diag_kind='kde',
             plot_kws={'alpha': 0.5, 's': 15})
plt.suptitle('Pairplot - Key Performance Indicators by Win Category', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 7.4 Average Win Percentage by School

In [ ]:
school_avg = df.groupby('school')['win_percentage'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
school_avg.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Average Win Percentage by School', fontsize=14, fontweight='bold')
plt.xlabel('School')
plt.ylabel('Avg Win Percentage')
plt.axhline(y=df['win_percentage'].mean(), color='red', linestyle='--', label='Overall Mean')
plt.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 7.5 Points vs Win Percentage (Scatter Plot)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Points vs Win%
sns.scatterplot(data=df, x='points', y='win_percentage', hue='win_category',
                palette={'High': 'green', 'Low': 'red'}, alpha=0.5, ax=axes[0])
axes[0].set_title('Points vs Win Percentage', fontweight='bold')

# FG% vs Win%
sns.scatterplot(data=df, x='field_goal_percentage', y='win_percentage', hue='win_category',
                palette={'High': 'green', 'Low': 'red'}, alpha=0.5, ax=axes[1])
axes[1].set_title('Field Goal % vs Win Percentage', fontweight='bold')

plt.tight_layout()
plt.show()

### 7.6 Top vs Bottom Teams Comparison

In [ ]:
# Compare rank 1-3 vs bottom ranked teams
top_teams = df[df['rank'] <= 3]
bottom_teams = df[df['rank'] >= 8]

compare_cols = ['points', 'field_goal_percentage', 'assists', 'turnovers', 'simple_rating']
comparison = pd.DataFrame({
    'Top Teams (Rank 1-3)': top_teams[compare_cols].mean(),
    'Bottom Teams (Rank 8+)': bottom_teams[compare_cols].mean()
}).round(2)

comparison.plot(kind='bar', figsize=(10, 5), color=['green', 'red'], edgecolor='black')
plt.title('Top Ranked vs Bottom Ranked Teams - Key Stats', fontsize=14, fontweight='bold')
plt.ylabel('Average Value')
plt.xticks(rotation=45, ha='right')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

comparison

---
## 8. Hypothesis Framing & Testing

We frame 5 hypotheses relevant to identifying key performance drivers for basketball team success.

**Significance Level: α = 0.05**

| # | Hypothesis | Test Method |
|---|-----------|-------------|
| H1 | Higher field goal percentage leads to higher win percentage | Pearson Correlation |
| H2 | Teams with positive net rating win significantly more than teams with negative net rating | Independent t-test |
| H3 | Assist-to-turnover ratio is significantly correlated with win percentage | Pearson Correlation |
| H4 | Home win percentage is significantly higher than away win percentage | Paired t-test |
| H5 | There is a significant difference in points scored between high-win and low-win teams | Independent t-test |

### Hypothesis 1: Higher Field Goal % → Higher Win %
- **H₀ (Null):** There is no significant correlation between field goal percentage and win percentage.
- **H₁ (Alternative):** There is a significant positive correlation between field goal percentage and win percentage.
- **Test:** Pearson Correlation

In [ ]:
corr, p_value = pearsonr(df['field_goal_percentage'], df['win_percentage'])

print("=" * 60)
print("HYPOTHESIS 1: Field Goal % vs Win Percentage")
print("=" * 60)
print(f"Pearson Correlation Coefficient: {corr:.4f}")
print(f"P-value: {p_value:.2e}")
print(f"Significance Level (α): 0.05")
print("-" * 60)
if p_value < 0.05:
    print("✅ RESULT: Reject H₀ — There IS a significant positive correlation.")
    print(f"   Teams with higher field goal % tend to have higher win %.")
else:
    print("❌ RESULT: Fail to reject H₀ — No significant correlation found.")
print("=" * 60)

### Hypothesis 2: Positive Net Rating → More Wins
- **H₀ (Null):** There is no significant difference in win percentage between teams with positive and negative net ratings.
- **H₁ (Alternative):** Teams with positive net rating have significantly higher win percentage.
- **Test:** Independent Samples t-test

In [ ]:
positive_net = df[df['net_rating'] > 0]['win_percentage']
negative_net = df[df['net_rating'] <= 0]['win_percentage']

t_stat, p_value = ttest_ind(positive_net, negative_net)

print("=" * 60)
print("HYPOTHESIS 2: Net Rating vs Win Percentage")
print("=" * 60)
print(f"Positive Net Rating - Mean Win%: {positive_net.mean():.4f} (n={len(positive_net)})")
print(f"Negative Net Rating - Mean Win%: {negative_net.mean():.4f} (n={len(negative_net)})")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.2e}")
print(f"Significance Level (α): 0.05")
print("-" * 60)
if p_value < 0.05:
    print("✅ RESULT: Reject H₀ — Teams with positive net rating win significantly more.")
else:
    print("❌ RESULT: Fail to reject H₀ — No significant difference found.")
print("=" * 60)

### Hypothesis 3: Assist-to-Turnover Ratio Correlates with Win %
- **H₀ (Null):** There is no significant correlation between assist-to-turnover ratio and win percentage.
- **H₁ (Alternative):** There is a significant correlation between assist-to-turnover ratio and win percentage.
- **Test:** Pearson Correlation

In [ ]:
# Drop any inf values from the ratio
valid_data = df[np.isfinite(df['assist_turnover_ratio'])]

corr, p_value = pearsonr(valid_data['assist_turnover_ratio'], valid_data['win_percentage'])

print("=" * 60)
print("HYPOTHESIS 3: Assist-to-Turnover Ratio vs Win Percentage")
print("=" * 60)
print(f"Pearson Correlation Coefficient: {corr:.4f}")
print(f"P-value: {p_value:.2e}")
print(f"Significance Level (α): 0.05")
print("-" * 60)
if p_value < 0.05:
    print("✅ RESULT: Reject H₀ — Assist-to-turnover ratio IS significantly correlated with win %.")
    print(f"   Better ball handling (higher A/T ratio) is linked to more wins.")
else:
    print("❌ RESULT: Fail to reject H₀ — No significant correlation found.")
print("=" * 60)

### Hypothesis 4: Home Win % > Away Win % (Home Court Advantage)
- **H₀ (Null):** There is no significant difference between home and away win percentages.
- **H₁ (Alternative):** Home win percentage is significantly higher than away win percentage.
- **Test:** Paired t-test (same teams, different conditions)

In [ ]:
# Filter out rows where either is NaN or inf
valid = df[np.isfinite(df['home_win_pct']) & np.isfinite(df['away_win_pct'])]

t_stat, p_value = ttest_rel(valid['home_win_pct'], valid['away_win_pct'])

print("=" * 60)
print("HYPOTHESIS 4: Home Court Advantage")
print("=" * 60)
print(f"Mean Home Win%: {valid['home_win_pct'].mean():.4f}")
print(f"Mean Away Win%: {valid['away_win_pct'].mean():.4f}")
print(f"Difference: {(valid['home_win_pct'].mean() - valid['away_win_pct'].mean()):.4f}")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.2e}")
print(f"Significance Level (α): 0.05")
print("-" * 60)
if p_value < 0.05:
    print("✅ RESULT: Reject H₀ — Home court advantage is statistically significant.")
    print("   Teams perform significantly better at home than away.")
else:
    print("❌ RESULT: Fail to reject H₀ — No significant home court advantage found.")
print("=" * 60)

### Hypothesis 5: High-Win Teams Score More Points
- **H₀ (Null):** There is no significant difference in points scored between high-win and low-win teams.
- **H₁ (Alternative):** High-win teams score significantly more points than low-win teams.
- **Test:** Independent Samples t-test

In [ ]:
high_win_points = df[df['win_category'] == 'High']['points']
low_win_points = df[df['win_category'] == 'Low']['points']

t_stat, p_value = ttest_ind(high_win_points, low_win_points)

print("=" * 60)
print("HYPOTHESIS 5: Points Scored - High vs Low Win Teams")
print("=" * 60)
print(f"High-Win Teams - Mean Points: {high_win_points.mean():.2f} (n={len(high_win_points)})")
print(f"Low-Win Teams  - Mean Points: {low_win_points.mean():.2f} (n={len(low_win_points)})")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.2e}")
print(f"Significance Level (α): 0.05")
print("-" * 60)
if p_value < 0.05:
    print("✅ RESULT: Reject H₀ — High-win teams score significantly more points.")
else:
    print("❌ RESULT: Fail to reject H₀ — No significant difference in scoring.")
print("=" * 60)

### Hypothesis Testing Summary

In [ ]:
summary = pd.DataFrame({
    'Hypothesis': [
        'H1: FG% → Win%',
        'H2: Net Rating → Wins',
        'H3: A/T Ratio → Win%',
        'H4: Home > Away Win%',
        'H5: High-Win Teams Score More'
    ],
    'Test Used': [
        'Pearson Correlation',
        'Independent t-test',
        'Pearson Correlation',
        'Paired t-test',
        'Independent t-test'
    ],
    'Significance (α=0.05)': [
        '✅ Significant' if pearsonr(df['field_goal_percentage'], df['win_percentage'])[1] < 0.05 else '❌ Not Significant',
        '✅ Significant' if ttest_ind(df[df['net_rating']>0]['win_percentage'], df[df['net_rating']<=0]['win_percentage'])[1] < 0.05 else '❌ Not Significant',
        '✅ Significant' if pearsonr(valid_data['assist_turnover_ratio'], valid_data['win_percentage'])[1] < 0.05 else '❌ Not Significant',
        '✅ Significant' if ttest_rel(valid['home_win_pct'], valid['away_win_pct'])[1] < 0.05 else '❌ Not Significant',
        '✅ Significant' if ttest_ind(high_win_points, low_win_points)[1] < 0.05 else '❌ Not Significant'
    ]
})

summary

---
## 9. Classification Model & Confusion Matrix

We build a **Random Forest Classifier** to predict whether a team will have a **High** or **Low** win season based on their performance stats. This helps identify the key drivers of success.

In [ ]:
# Prepare features and target
feature_cols = ['points', 'opponent_points', 'field_goal_percentage', 'free_throw_percentage',
                'total_rebounds', 'assists', 'steals', 'blocks', 'turnovers',
                'offensive_rating', 'defensive_rating', 'simple_rating',
                'point_diff', 'assist_turnover_ratio']

X = df[feature_cols]
y = df['win_category']

# Handle any remaining inf/nan
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.median())

# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# Train Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predictions
y_pred = rf_model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")

### 9.1 Confusion Matrix

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=['High', 'Low'])

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['High Win', 'Low Win'])
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title('Confusion Matrix - Win Category Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['High Win', 'Low Win']))

### 9.2 Feature Importance
Which stats matter most for predicting winning seasons?

In [ ]:
# Feature Importance
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=True)

plt.figure(figsize=(10, 7))
importances.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Feature Importance - Key Drivers of Win Category', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print("\nTop 5 Most Important Features:")
importances.sort_values(ascending=False).head()

---
## 10. Final Observations & Conclusions

### Key Findings:

1. **Field Goal Percentage is a key driver:** Teams with higher shooting accuracy have significantly higher win percentages (H1 confirmed).

2. **Net Rating strongly predicts success:** Teams with a positive net rating (scoring more than opponents) win significantly more games (H2 confirmed).

3. **Ball handling matters:** The assist-to-turnover ratio is a significant indicator of team success — efficient ball movement leads to more wins (H3 confirmed).

4. **Home court advantage exists:** Teams win at a significantly higher rate at home compared to away games (H4 confirmed).

5. **Scoring volume differentiates winners:** High-win teams score significantly more points per game than low-win teams (H5 confirmed).

6. **Top performance drivers (from model):** Simple rating, point differential, and offensive/defensive ratings are the most important features for predicting season success.

### Implications for Coaching & Recruitment:
- **Prioritize shooting efficiency** over volume — field goal percentage matters more than attempts.
- **Reduce turnovers** — the assist-to-turnover ratio is a key differentiator between winning and losing teams.
- **Focus on defense:** Defensive rating and limiting opponent scoring are as important as offensive output.
- **Leverage home court:** Schedule strategically, as home performance significantly exceeds away performance.
- **Recruit well-rounded players** who contribute to both ends of the floor (points + rebounds + assists).

In [ ]:
print("="*60)
print("ANALYSIS COMPLETE")
print("="*60)
print(f"Dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Hypotheses Tested: 5")
print(f"Classification Model Accuracy: {accuracy*100:.1f}%")
print(f"Top Performance Driver: {importances.sort_values(ascending=False).index[0]}")
print("="*60)